In [6]:
import pandas as pd
from datetime import datetime, timedelta
from io import StringIO

def time_to_seconds(time_str):
    t = datetime.strptime(time_str.strip(), "%H:%M:%S.%f")
    return t.hour * 3600 + t.minute * 60 + t.second + t.microsecond / 1e6

def seconds_to_time(seconds):
    return str(timedelta(seconds=seconds))

def compute_metadata(base_metadata, df, num_speakers=None):
    """Update metadata for duration, number of speakers, and number of segments."""
    meta = []
    for line in base_metadata:
        if line.lower().startswith("# duration:"):
            # compute duration from df
            start_time = time_to_seconds(df.iloc[0]['Start Time'])
            end_time = time_to_seconds(df.iloc[-1]['End Time'])
            duration = seconds_to_time(end_time - start_time)
            meta.append(f"# Duration: {duration}\n")
        elif line.lower().startswith("# number of speakers:") and num_speakers is not None:
            meta.append(f"# Number of Speakers: {num_speakers}\n")
        elif line.lower().startswith("# number of segments:"):
            meta.append(f"# Number of Segments: {len(df)}\n")
        else:
            meta.append(line)
    return meta

def conditional_role_mapping(df, mapping_dict):
    """Apply mapping rules to update Speaker Role based on conditions."""
    df = df.copy()

    # Apply Speaker → Speaker Role mapping
    if 'Speaker' in mapping_dict:
        for speaker_val, new_role in mapping_dict['Speaker'].items():
            df.loc[df['Speaker'] == speaker_val, 'Speaker Role'] = new_role

    # Apply Speaker Role remapping
    if 'Speaker Role' in mapping_dict:
        for old_role, new_role in mapping_dict['Speaker Role'].items():
            df.loc[df['Speaker Role'] == old_role, 'Speaker Role'] = new_role

    return df

def save_with_metadata(df, path, metadata, num_speakers=None):
    """Save a dataframe to file with updated metadata."""
    updated_metadata = compute_metadata(metadata, df, num_speakers=num_speakers)
    with open(path, 'w', encoding='utf-8') as f:
        for line in updated_metadata:
            f.write(line)
        df.to_csv(f, sep='|', index=False, header=False)

def transcript_splitter(file_path, mapped_file, bailey_file, ali_file, combined_file, mapping_dict):
    # --- Read input file ---
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    metadata = [line for line in lines if line.strip().startswith('#')]
    content = [line for line in lines if not line.strip().startswith('#')]

    df = pd.read_csv(StringIO(''.join(content)), sep='|', header=None, engine='python')
    df.columns = ['Start Time', 'End Time', 'Speaker', 'Speaker Role', 'Transcription']
    df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

    # --- Step 1: Apply mapping ---
    df_mapped = conditional_role_mapping(df, mapping_dict)
    # df_mapped = df

    # --- Step 2: Save mapped full transcript (keep original metadata) ---
    save_with_metadata(df_mapped, mapped_file, metadata)

    # --- Step 3: Separate Bailey and Ali ---
    df_bailey = df_mapped[df_mapped['Speaker Role'].str.lower() == 'bailey'].copy()
    df_ali = df_mapped[df_mapped['Speaker Role'].str.lower() == 'ali'].copy()
    # df_bailey = df_mapped[df_mapped['Speaker'].str.lower() == 'speaker 1'].copy()
    # df_ali = df_mapped[df_mapped['Speaker'].str.lower() == 'speaker 2'].copy()

    df_bailey['Speaker'] = 'Bailey'
    df_ali['Speaker'] = 'Ali'

    save_with_metadata(df_bailey, bailey_file, metadata, num_speakers=1)
    save_with_metadata(df_ali, ali_file, metadata, num_speakers=1)

    # --- Step 4: Create combined transcript ---
    df_combined = pd.concat([df_bailey, df_ali]).sort_values(by=['Start Time'])
    save_with_metadata(df_combined, combined_file, metadata, num_speakers=2)

    print(f"✅ Saved mapped full transcript → {mapped_file}")
    print(f"✅ Saved Bailey transcript → {bailey_file}")
    print(f"✅ Saved Ali transcript → {ali_file}")
    print(f"✅ Saved Combined transcript → {combined_file}")

In [1]:
import re
import argparse
from difflib import SequenceMatcher
import pandas as pd
from datetime import datetime, timedelta
from io import StringIO
from typing import Dict, List, Tuple, Set

# ===================== Time & metadata helpers =====================

def time_to_seconds(time_str: str) -> float:
    t = datetime.strptime(time_str.strip(), "%H:%M:%S.%f")
    return t.hour * 3600 + t.minute * 60 + t.second + t.microsecond / 1e6

def seconds_to_time(seconds: float) -> str:
    return str(timedelta(seconds=seconds))

def compute_metadata(base_metadata: List[str], df: pd.DataFrame, num_speakers: int = None) -> List[str]:
    meta = []
    for line in base_metadata:
        low = line.lower()
        if low.startswith("# duration:"):
            start_time = time_to_seconds(df.iloc[0]['Start Time']) if not df.empty else 0
            end_time = time_to_seconds(df.iloc[-1]['End Time']) if not df.empty else 0
            duration = seconds_to_time(max(0, end_time - start_time))
            meta.append(f"# Duration: {duration}\n")
        elif low.startswith("# number of speakers:") and num_speakers is not None:
            meta.append(f"# Number of Speakers: {num_speakers}\n")
        elif low.startswith("# number of segments:"):
            meta.append(f"# Number of Segments: {len(df)}\n")
        else:
            meta.append(line)
    return meta

def save_with_metadata(df: pd.DataFrame, path: str, metadata: List[str], num_speakers: int = None) -> None:
    updated_metadata = compute_metadata(metadata, df, num_speakers=num_speakers)
    with open(path, 'w', encoding='utf-8') as f:
        for line in updated_metadata:
            f.write(line)
        if not df.empty:
            df.to_csv(f, sep='|', index=False, header=False)

# ===================== Self-intro detection (fuzzy) =====================

TARGET_NAMES = ["Ali", "Bailey"]
TARGET_NAMES = ["Rowan", "SAM"]
TARGET_NAMES = ["Ali", "Rowan"]

# Expand with variants you've seen in your ASR
# NAME_ALIASES = {
#     "ali": {
#         "alli","ally","allie","aly","alley","alee","alie","allee",
#         "aalie","aali","ale","ely","elie","aley","ali"
#     },
#     "rowan": {
#         "roan","rowen","rowenn","roann","rohan","rohanne",
#         "rowynn","rowin","rowe","roen","rowan"
#     },
# }
NAME_ALIASES = {
    "ali": {
        "alli","ally","allie","aly","alley","alee","alie","allee",
        "aalie","aali","ale","ely","elie","aley","ali"
    },
    # "bailey": {
    #     "baily","baile","bailee","baylee","baileigh","beyley","baley",
    #     "baly","beyli","beylie","baely","baele","bailli","bailey"
    # },
    # Trust
    "rowan": {
        # direct & minor vowel variants
        "rowan","rowen","rowin","rowyn","rowon","rowane",
        # shortened / phonetic-ish
        "roan","rouan","roowan","ruwan",
        # doubled letters / endings
        "rowann","rowanne","rowynne",
        # casual / with punctuation captured by normalization
        "rowan.","rowin'"
    },
    # "sam": {
    #     # direct & small variations
    #     "sam","samm","sam.","samy","samy.","sami",
    #     # diminutives / nicknames
    #     "sammy","sammi","sammie","sammey",
    #     # common full-name mappings to Sam
    #     "samuel","samual","samwel","samwell"
    # },
}

HONORIFICS = {"mr", "mrs", "ms", "miss", "dr", "prof", "sir", "madam", "mx"}

# Intro patterns (first-person self-intro). Capture name in group 'name'.
INTRO_PATTERNS = [
    r"\b(?:i am|i'm|im)\s+(?P<name>[a-z][a-z\-']+(?:\s+[a-z][a-z\-']+)*)\b",
    r"\bmy\s+name\s+is\s+(?P<name>[a-z][a-z\-']+(?:\s+[a-z][a-z\-']+)*)\b",
    r"\bthis\s+is\s+(?P<name>[a-z][a-z\-']+(?:\s+[a-z][a-z\-']+)*)\b",
    r"\b(?P<name>[a-z][a-z\-']+)\s+here\b",
    r"\b(?:hey|hi|hello)[,!\.\s]*(?:i'?m|i am)?\s*(?P<name>[a-z][a-z\-']+)\b",
]

def _normalize_token(s: str) -> str:
    s = s.lower()
    s = re.sub(r"[^\w\s'-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    # collapse long repeated letters: "baaiiiley" -> "bailey"
    s = re.sub(r"(.)\1{2,}", r"\1\1", s)
    parts = s.split()
    if parts and parts[0] in HONORIFICS:
        parts = parts[1:]
    return parts[0] if parts else ""

def _sim(a: str, b: str) -> float:
    return SequenceMatcher(None, a, b).ratio()

def fuzzy_match_name(cand: str, target_names=TARGET_NAMES, threshold: float = 0.80):
    """
    Return canonical target name ('Ali'/'Bailey') if 'cand' is a close match; else None.
    Prefers exact, then alias, then fuzzy.
    """
    if not cand:
        return None

    c = _normalize_token(cand)
    if not c:
        return None

    # exact
    for t in target_names:
        if c == t.lower():
            return t

    # alias
    for t in target_names:
        if c in NAME_ALIASES.get(t.lower(), set()):
            return t

    # fuzzy
    best_t, best_s = None, 0.0
    for t in target_names:
        s = _sim(c, t.lower())
        if s > best_s:
            best_t, best_s = t, s
    if best_s >= threshold:
        return best_t
    return None
    
def detect_self_intro_name(text: str) -> str:
    """Return normalized first-token name if the speaker introduces themselves; else None."""
    t = text.lower()
    for pat in INTRO_PATTERNS:
        m = re.search(pat, t, flags=re.IGNORECASE)
        if m:
            raw = m.group("name")
            return _normalize_token(raw)
    return None

# ===================== Utilities for Speaker 1 / Speaker 2 =====================

def norm_speaker_label(s: str) -> str:
    """Normalize 'Speaker 1', 'speaker1', 'SPEAKER 2' -> 'speaker 1' / 'speaker 2'."""
    s = (s or "").strip().lower()
    s = re.sub(r"\s+", " ", s)
    s = s.replace("speaker", "speaker ")
    s = re.sub(r"\s+", " ", s).strip()
    # coalesce forms like 'speaker  1' or 'speaker1'
    m = re.match(r"^speaker\s*([0-9]+)$", s.replace(" ", ""))
    if m:
        return f"speaker {m.group(1)}"
    m2 = re.match(r"^speaker\s+([0-9]+)$", s)
    if m2:
        return f"speaker {m2.group(1)}"
    return s

PRIMARY_PAIR = ["speaker 1", "speaker 2"]

# ===================== Role inference with Speaker1/2 counterpart rule =====================
def infer_roles_by_self_intro(
    df: pd.DataFrame,
    target_names=TARGET_NAMES,
    fuzzy_threshold: float = 0.80
) -> Tuple[Dict[str, str], List[str], Set[str]]:
    """
    Self-intros (exact/alias/fuzzy) assign Ali/Bailey.
    If exactly one role is detected, assign the other role to the OTHER of {Speaker 1, Speaker 2}.
    If at least one role exists after that, remaining unassigned -> RA.
    If neither role is detected, do NOT assign RA; leave roles unchanged.

    Returns (speaker_role_map, logs_list, role_taken_set).
    """
    logs: List[str] = []
    speaker_to_role: Dict[str, str] = {}
    role_taken: Set[str] = set()

    # Build a map from normalized -> original speaker labels (to write back)
    uniq_speakers = list(df["Speaker"].unique())
    norm_to_orig = {norm_speaker_label(s): s for s in uniq_speakers}
    orig_to_norm = {v: k for k, v in norm_to_orig.items()}

    # 1) First pass: self-introductions
    for _, row in df.iterrows():
        spk_orig = row["Speaker"]
        spk_norm = orig_to_norm.get(spk_orig, norm_speaker_label(spk_orig))
        intro_name = detect_self_intro_name(row["Transcription"])
        if not intro_name:
            continue

        canonical = None
        reason = None

        # exact / alias / fuzzy
        if intro_name in {t.lower() for t in target_names}:
            canonical = next(t for t in target_names if t.lower() == intro_name)
            reason = "exact"
        if canonical is None:
            for t in target_names:
                if intro_name in NAME_ALIASES.get(t.lower(), set()):
                    canonical = t
                    reason = "alias"
                    break
        if canonical is None:
            cand = fuzzy_match_name(intro_name, target_names=target_names, threshold=fuzzy_threshold)
            if cand:
                canonical = cand
                reason = f"fuzzy≥{fuzzy_threshold:.2f}"

        if canonical:
            if canonical in role_taken and speaker_to_role.get(spk_orig) != canonical:
                logs.append(f"[Conflict] '{canonical}' already assigned; speaker '{spk_orig}' also claimed via {reason} ('{intro_name}'). Ignored.")
                continue
            speaker_to_role[spk_orig] = canonical
            role_taken.add(canonical)
            logs.append(f"[Assign] Speaker '{spk_orig}' claimed '{canonical}' via {reason} ('{intro_name}').")

    # 2) Counterpart rule for exactly one detected role → other of {Speaker 1, Speaker 2}
    if len(role_taken) == 1:
        detected = next(iter(role_taken))
        counterpart = next(t for t in target_names if t != detected)

        # Which of Speaker 1/2 is already assigned?
        primary_present = [p for p in PRIMARY_PAIR if p in norm_to_orig]
        if primary_present:
            # Find which primary (if any) already has the detected role
            primary_with_detected = None
            for p in PRIMARY_PAIR:
                orig_label = norm_to_orig.get(p)
                if orig_label and speaker_to_role.get(orig_label) == detected:
                    primary_with_detected = p
                    break

            if primary_with_detected:
                    # Assign counterpart to the OTHER primary (if present & unassigned)
                    other_primary = PRIMARY_PAIR[1] if primary_with_detected == PRIMARY_PAIR[0] else PRIMARY_PAIR[0]
                    other_orig = norm_to_orig.get(other_primary)
                    if other_orig and other_orig not in speaker_to_role:
                        speaker_to_role[other_orig] = counterpart
                        role_taken.add(counterpart)
                        logs.append(f"[Counterpart] Only '{detected}' detected. Assigned '{counterpart}' to '{other_orig}' (the other of Speaker 1/2).")
            else:
                # Detected role belongs to a non-primary speaker. Assign counterpart to the first available primary.
                for p in PRIMARY_PAIR:
                    other_orig = norm_to_orig.get(p)
                    if other_orig and other_orig not in speaker_to_role:
                        speaker_to_role[other_orig] = counterpart
                        role_taken.add(counterpart)
                        logs.append(f"[Counterpart] Only '{detected}' detected (non-primary). Assigned '{counterpart}' to '{other_orig}' (preferred Speaker 1/2).")
                        break
        else:
            logs.append("[Counterpart] Speaker 1/2 labels not present; skipped primary-pair counterpart rule.")

    # 3) Only assign RA if at least one role is now taken
    if role_taken:
        for spk in df["Speaker"].unique():
            if spk not in speaker_to_role:
                speaker_to_role[spk] = "RA"
                logs.append(f"[Unresolved] Could not infer Ali/Bailey for speaker '{spk}'. Marked as 'RA'.")
    else:
        logs.append("[Notice] Neither 'Ali' nor 'Bailey' was detected (exact/alias/fuzzy). No 'RA' will be assigned; roles remain unchanged.")

    return speaker_to_role, logs, role_taken


def canonicalize_other_speakers(
    df: pd.DataFrame,
    primary_pair: List[str] = PRIMARY_PAIR,
    start_from: int = 3,
    time_col: str = "Start Time"
) -> Tuple[pd.DataFrame, Dict[str, str], List[str]]:
    """
    Convert any non-primary speaker labels (e.g., 'A','B','Guest','Erin') into
    'speaker N' starting at N = max(existing speaker numbers, start_from-1) + 1.
    Keeps existing numbered 'speaker k' labels as-is (including 1 and 2).
    The order of assignment is based on first appearance time.

    Returns: (df_updated, mapping_dict, logs)
    """
    logs = []

    def _num_from_label(s: str) -> int | None:
        s_ = (s or "").strip().lower()
        s_ = re.sub(r"\s+", " ", s_)
        # accept 'speaker 3' or 'speaker3'
        m = re.match(r"^speaker\s*([0-9]+)$", s_.replace(" ", ""))
        if m:
            return int(m.group(1))
        m2 = re.match(r"^speaker\s+([0-9]+)$", s_)
        if m2:
            return int(m2.group(1))
        return None

    # Collect already-used numeric labels
    used_nums = set()
    for s in df["Speaker"].unique():
        n = _num_from_label(s)
        if n is not None:
            used_nums.add(n)

    next_num = max(used_nums) + 1 if used_nums else start_from
    # Ensure next_num is at least start_from
    next_num = max(next_num, start_from)

    # Determine first-appearance order
    # If time is malformed, treat as large to push to the end
    def _safe_seconds(x: str) -> float:
        try:
            return time_to_seconds(x)
        except Exception:
            return 10**12

    first_seen = (
        df.assign(_t=df[time_col].apply(_safe_seconds))
          .groupby("Speaker", as_index=False)["_t"].min()
          .sort_values("_t")
    )

    # Build mapping for non-primary, non-numbered labels
    primary_norm = {norm_speaker_label(p) for p in primary_pair}
    mapping: Dict[str, str] = {}

    for spk in first_seen["Speaker"]:
        spk_norm = norm_speaker_label(spk)
        already_numbered = _num_from_label(spk) is not None
        if spk_norm in primary_norm or already_numbered:
            continue
        # Assign a fresh numeric label
        while next_num in used_nums:
            next_num += 1
        mapping[spk] = f"speaker {next_num}"
        used_nums.add(next_num)
        logs.append(f"[Canon] '{spk}' → 'speaker {next_num}'")
        next_num += 1

    if mapping:
        df = df.copy()
        df["Speaker"] = df["Speaker"].map(lambda s: mapping.get(s, s))
    else:
        logs.append("[Canon] No non-primary labels required renumbering.")

    return df, mapping, logs
    
# ===================== Main: read, infer, write =====================

def transcript_splitter_by_intro(
    file_path: str,
    mapped_file: str,
    bailey_file: str,
    ali_file: str,
    combined_file: str,
    role_names=TARGET_NAMES,
    fuzzy_threshold: float = 0.80,
) -> None:
    """
    Reads one transcript file with format:
      # Format: Start Time | End Time | Speaker | Speaker Role | Transcription
    Infers roles ONLY from self-introductions (robust to ASR via alias/fuzzy).
    Others → 'RA'. Writes mapped, role-only, and combined files.
    """
    # --- Read input file ---
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    metadata = [line for line in lines if line.strip().startswith('#')]
    content = [line for line in lines if not line.strip().startswith('#')]

    if not content:
        raise ValueError("No transcript content found (only metadata?).")

    df = pd.read_csv(StringIO(''.join(content)), sep='|', header=None, engine='python')
    df.columns = ['Start Time', 'End Time', 'Speaker', 'Speaker Role', 'Transcription']
    df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

    # --- Canonicalize non-primary speakers to 'speaker 3', 'speaker 4', ... ---
    df, canon_map, canon_logs = canonicalize_other_speakers(df, primary_pair=PRIMARY_PAIR, start_from=3)

    # ---- Logs
    print("\n=== Canonicalization Log (Other speakers → speaker #) ===")
    for line in canon_logs:
        print(line)
    print("=== End Canon Log ===\n")

    # --- Infer roles by self-introduction (exact/alias/fuzzy) ---
    speaker_role_map, logs, role_taken = infer_roles_by_self_intro(
        df, target_names=role_names, fuzzy_threshold=fuzzy_threshold
    )

    # Overwrite roles ONLY where we inferred something; otherwise keep original
    df_mapped = df.copy()
    if role_taken:
        # At least one of Ali/Bailey found: apply all mappings (including RA for unassigned)
        df_mapped['Speaker Role'] = df_mapped.apply(
            lambda r: speaker_role_map.get(r['Speaker'], r['Speaker Role']),
            axis=1
        )
    else:
        # Neither role found: leave roles exactly as they were
        logs.append("[Action] Leaving all 'Speaker Role' values unchanged in the mapped file (no Ali/Bailey detected).")

    # --- Save mapped full transcript ---
    save_with_metadata(df_mapped, mapped_file, metadata, num_speakers=len(df_mapped['Speaker'].unique()))

    # --- Split per role names (Ali / Bailey) ---
    role_a, role_b = role_names  # Ali, Bailey
    df_a = df_mapped[df_mapped['Speaker Role'].str.lower() == role_a.lower()].copy()
    df_b = df_mapped[df_mapped['Speaker Role'].str.lower() == role_b.lower()].copy()

    if not df_a.empty:
        # df_a['Speaker'] = role_a
        save_with_metadata(df_a, ali_file, metadata, num_speakers=1)
    else:
        df_a = df_mapped[df_mapped['Speaker'].str.lower() == 'speaker 1'].copy()
        save_with_metadata(df_a, ali_file, metadata, num_speakers=1)

    if not df_b.empty:
        # df_b['Speaker'] = role_b
        save_with_metadata(df_b, bailey_file, metadata, num_speakers=1)
    else:
        df_b = df_mapped[df_mapped['Speaker'].str.lower() == 'speaker 2'].copy()
        save_with_metadata(df_b, bailey_file, metadata, num_speakers=1)

    # --- Combined (only Ali+Bailey) ---
    df_combined = pd.concat([df_a, df_b], ignore_index=True).sort_values(by=['Start Time'])
    save_with_metadata(
        df_combined,
        combined_file,
        metadata,
        num_speakers=len(df_combined['Speaker'].unique()) if not df_combined.empty else 0
    )
    
    # ---- Logs
    print("\n=== Role Inference Log (Self-introduction with fuzzy/alias) ===")
    for line in logs:
        print(line)
    print("=== End Log ===\n")

    print(f"✅ Saved mapped full transcript → {mapped_file}")
    print(f"✅ Saved {role_b} transcript → {bailey_file}")
    print(f"✅ Saved {role_a} transcript → {ali_file}")
    print(f"✅ Saved Combined transcript → {combined_file}")

In [5]:
from pathlib import Path

# --- configure these two ---
BASE_ROOT = Path("./Test/GSGS#2_W24")          # folder that contains GS1-FtF10A, GS1-FtF20A, ...
VERSION = ""                                # e.g., "-v2" if you need a suffix; else leave ""

def process_all(base_root: Path, version: str = ""):
    subdirs = [p for p in base_root.iterdir() if p.is_dir()]
    processed, skipped = 0, []

    for sub in sorted(subdirs):
        name = sub.name                         # e.g., "GS1-FtF10A"
        full_dir = sub / "full"
        if not full_dir.is_dir():
            skipped.append((name, "missing 'full/'"))
            continue

        # Preferred: exact match {name}{version}_full.txt
        preferred = full_dir / f"{name}{version}_full.txt"
        if preferred.exists():
            input_file = preferred
            base_out = f"{name}{version}"
        else:
            # Fallback: any *_full.txt inside full/
            matches = list(full_dir.glob("*_full.txt"))
            if not matches:
                skipped.append((name, "no '*_full.txt' found"))
                continue
            input_file = matches[0]
            # derive the base name for outputs by stripping the trailing '_full'
            stem = input_file.stem
            base_out = stem[:-5] if stem.endswith("_full") else stem

        try:
            # --- your function call ---
            transcript_splitter_by_intro(
                input_file,
                full_dir / f"{base_out}_full.txt",
                full_dir / f"{base_out}_single_B.txt",
                full_dir / f"{base_out}_single_A.txt",
                full_dir / f"{base_out}_dyad.txt",
            )
            processed += 1
            print(f"[OK] {name}")
        except Exception as e:
            skipped.append((name, f"error: {e!s}"))

    print(f"\nDone: {processed}/{len(subdirs)} processed.")
    if skipped:
        print("Skipped:")
        for n, why in skipped:
            print(f"  - {n}: {why}")

process_all(BASE_ROOT, VERSION)

/tmp/ipykernel_824/57494870.py:389: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/tmp/ipykernel_824/57494870.py:389: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/tmp/ipykernel_824/57494870.py:389: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/tmp/ipykernel_824/57494870.py:389: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/tmp/ipykernel_824/57494870.py:389: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/tmp/ipykernel_824/57494870.py:389: FutureWarning: DataFrame.appl


=== Canonicalization Log (Other speakers → speaker #) ===
[Canon] No non-primary labels required renumbering.
=== End Canon Log ===


=== Role Inference Log (Self-introduction with fuzzy/alias) ===
[Assign] Speaker 'Speaker 1' claimed 'Ali' via exact ('ali').
[Counterpart] Only 'Ali' detected. Assigned 'Rowan' to 'Speaker 2' (the other of Speaker 1/2).
[Unresolved] Could not infer Ali/Bailey for speaker 'Speaker 3'. Marked as 'RA'.
=== End Log ===

✅ Saved mapped full transcript → Test/GSGS#2_W24/GS2W-VtV(SUSPECTED 2024 February 6 12 02 PM)(UNPINNED)/full/GS2W-VtV(SUSPECTED 2024 February 6 12 02 PM)(UNPINNED)_full.txt
✅ Saved Rowan transcript → Test/GSGS#2_W24/GS2W-VtV(SUSPECTED 2024 February 6 12 02 PM)(UNPINNED)/full/GS2W-VtV(SUSPECTED 2024 February 6 12 02 PM)(UNPINNED)_single_B.txt
✅ Saved Ali transcript → Test/GSGS#2_W24/GS2W-VtV(SUSPECTED 2024 February 6 12 02 PM)(UNPINNED)/full/GS2W-VtV(SUSPECTED 2024 February 6 12 02 PM)(UNPINNED)_single_A.txt
✅ Saved Combined transcript → Tes

/tmp/ipykernel_824/57494870.py:389: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/tmp/ipykernel_824/57494870.py:389: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/tmp/ipykernel_824/57494870.py:389: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/tmp/ipykernel_824/57494870.py:389: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/tmp/ipykernel_824/57494870.py:389: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/tmp/ipykernel_824/57494870.py:389: FutureWarning: DataFrame.appl

In [6]:
from pathlib import Path
# Deleting zone.identifier from directory
# Root folder containing all GS1-FtF* subfolders
# BASE_ROOT = Path("./Test/GSGS#2_W24")  # change this path if needed

def delete_zone_identifier_files(base_root: Path):
    count = 0
    for file in base_root.rglob("*Zone.Identifier*"):
        try:
            file.unlink()
            print(f"✅ Deleted: {file}")
            count += 1
        except Exception as e:
            print(f"⚠️ Could not delete {file}: {e}")

    print(f"\nDone. Deleted {count} Zone.Identifier files.")

delete_zone_identifier_files(BASE_ROOT)


Done. Deleted 0 Zone.Identifier files.


In [7]:
from pathlib import Path
import shutil
# Moving files into 
# Root path (contains many GS1-FtF* folders)
# BASE_ROOT = Path("./Test/GSGS#2_W24")

# Top-level destination folders
DEST_FULL   = BASE_ROOT / "full"
DEST_SINGLE = BASE_ROOT / "single"
DEST_DYAD   = BASE_ROOT / "dyad"
for d in (DEST_FULL, DEST_SINGLE, DEST_DYAD):
    d.mkdir(parents=True, exist_ok=True)

def unique_dest(dest_dir: Path, filename: str) -> Path:
    """
    Avoid name collisions by appending _1, _2, ... if needed.
    """
    target = dest_dir / filename
    if not target.exists():
        return target
    stem, suffix = target.stem, target.suffix
    i = 1
    while True:
        cand = dest_dir / f"{stem}_{i}{suffix}"
        if not cand.exists():
            return cand
        i += 1

def collect_outputs(base_root: Path):
    moved = {"full": 0, "single": 0, "dyad": 0}
    # Only iterate immediate case folders (skip the top-level dest folders)
    for case_dir in sorted(p for p in base_root.iterdir() if p.is_dir()):
        if case_dir.name in {"full", "single", "dyad"}:
            continue

        full_dir = case_dir / "full"
        if not full_dir.is_dir():
            # skip cases without a full/ subdir
            continue

        for f in full_dir.glob("*.txt"):
            name = f.name.lower()

            if "_full" in name:
                dest = unique_dest(DEST_FULL, f.name)
                shutil.copy2(f, dest)
                moved["full"] += 1

            elif "_single_a" in name or "_single b" in name or "_single_b" in name:
                # handle both '_single_B' and potential space variants
                dest = unique_dest(DEST_SINGLE, f.name)
                shutil.copy2(f, dest)
                moved["single"] += 1

            elif "_dyad" in name:
                dest = unique_dest(DEST_DYAD, f.name)
                shutil.copy2(f, dest)
                moved["dyad"] += 1

    print("\nSummary:")
    for k, v in moved.items():
        print(f"  {k:<6}: {v} files")


collect_outputs(BASE_ROOT)


Summary:
  full  : 43 files
  single: 84 files
  dyad  : 42 files


# Convert to CSV file

In [5]:
import pandas as pd
# --- Read input file ---
# file_path = './Test/GS2F_95/GS2F/GS2F-VGC8A/full/GS2F-VGC8A_single_B.txt'
dir_ = 'GS1-VtV1A_single.txt'
file_path = f'./GS1-VtV1A_single.txt'
with open(file_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

metadata = [line for line in lines if line.strip().startswith('#')]
content = [line for line in lines if not line.strip().startswith('#')]

if not content:
    raise ValueError("No transcript content found (only metadata?).")

df = pd.read_csv(StringIO(''.join(content)), sep='|', header=None, engine='python')
df.columns = ['Start Time', 'End Time', 'Speaker', 'Speaker Role', 'Transcription']
df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

/tmp/ipykernel_671/2911633113.py:17: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)


In [6]:
df.head()

,Start Time,End Time,Speaker,Speaker Role,Transcription
0,00:00:07.580,00:00:08.650,Speaker 1,VtV1A,"Hi. I'm good, how are you?"
1,00:00:10.060,00:00:11.849,Speaker 1,VtV1A,Um. So how was your day so far?
2,00:00:17.339,00:00:25.849,Speaker 1,VtV1A,"Oh, that's good. That sounds like a very produ..."
3,00:00:27.500,00:00:28.170,Speaker 1,VtV1A,Thank you.
4,00:00:29.420,00:00:31.130,Speaker 1,VtV1A,"Um, I'm in the criminology department."


In [8]:
df.to_csv(f"./GS1-VtV1A_single.csv", index=False)

# Fixing issues manualy

In [6]:
import pandas as pd

roles = {
    'Speakers': {
        'Speaker 1':'Bailey',
        'Speaker 2': 'RA',
        'Speaker 3': 'Ali'
    },
    'Participants': [
        'Speaker 1', 
        'Speaker 3'
    ]
}
def read_content_file(path, roles):
    # --- Read input file ---
    # Test/GS2F/GS2F-VtV17AB(Unpinned)/full/GS2F-VtV17AB(Unpinned)_full.txt
    file_path = f'Test/GS2F/GS2F-VtV17AB(Unpinned)/full/GS2F-VtV17AB(Unpinned)_full.txt'
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    metadata = [line for line in lines if line.strip().startswith('#')]
    content = [line for line in lines if not line.strip().startswith('#')]
    
    if not content:
        raise ValueError("No transcript content found (only metadata?).")
    
    df = pd.read_csv(StringIO(''.join(content)), sep='|', header=None, engine='python')
    df.columns = ['Start Time', 'End Time', 'Speaker', 'Speaker Role', 'Transcription']
    for speaker, role in roles['Speakers'].items():
        df.loc[df['Speaker'] == speaker, 'Speaker Role'] = role

    df_ali, df_bailey = df.loc[df['Speaker Role'] == 'Ali'], df.loc[df['Speaker Role'] == 'Bailey']
    df_combined = pd.concat([df_bailey, df_ali]).sort_values(by=['Start Time'])
    save_with_metadata(df, "test_full.txt", metadata)
    save_with_metadata(df_combined, "test_dyad.txt", metadata, num_speakers=2)
    save_with_metadata(df_ali, "test_ali.txt", metadata, num_speakers=1)
    save_with_metadata(df_bailey, "test_bailey.txt", metadata, num_speakers=1)
    
    # df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

In [7]:
read_content_file("", roles)

# Renaming files in transcription

In [1]:
from pathlib import Path

ROOT = Path("Test/GSGS#2_W24")

for folder in ROOT.iterdir():
    if not folder.is_dir():
        continue

    # ----------------------------
    # 1. Rename folder (remove .MP4)
    # ----------------------------
    new_folder = folder
    if ".MP4" in folder.name:
        new_folder = folder.with_name(folder.name.replace(".MP4", ""))
        folder.rename(new_folder)
        print(f"Renamed folder: {folder.name} → {new_folder.name}")

    # ----------------------------
    # 2. Process full/ directory
    # ----------------------------
    full_dir = new_folder / "full"
    if not full_dir.exists():
        continue

    txt_files = list(full_dir.glob("*.txt"))
    if len(txt_files) != 1:
        print(f"Skipping {new_folder.name}: expected 1 txt file")
        continue

    txt_file = txt_files[0]

    # Rename text file
    new_txt_name = txt_file.name.replace(".MP4", "")
    new_txt_file = txt_file.with_name(new_txt_name)
    txt_file.rename(new_txt_file)
    print(f"Renamed file: {txt_file.name} → {new_txt_file.name}")

    # ----------------------------
    # 3. Fix # Title inside file
    # ----------------------------
    content = new_txt_file.read_text(encoding="utf-8")

    updated = []
    for line in content.splitlines():
        if line.startswith("# Title:"):
            line = line.replace(".MP4", "")
        updated.append(line)

    new_txt_file.write_text("\n".join(updated), encoding="utf-8")
    print(f"Updated title in: {new_txt_file.name}")


Renamed folder: GS2W-FtF24A.MP4 → GS2W-FtF24A
Renamed file: GS2W-FtF24A.MP4_full.txt → GS2W-FtF24A_full.txt
Updated title in: GS2W-FtF24A_full.txt
Renamed folder: GS2W-FtF22B.MP4 → GS2W-FtF22B
Renamed file: GS2W-FtF22B.MP4_full.txt → GS2W-FtF22B_full.txt
Updated title in: GS2W-FtF22B_full.txt
Renamed folder: GS2W-FtF30B.MP4 → GS2W-FtF30B
Renamed file: GS2W-FtF30B.MP4_full.txt → GS2W-FtF30B_full.txt
Updated title in: GS2W-FtF30B_full.txt
Renamed folder: GS2W-FtF35B.MP4 → GS2W-FtF35B
Renamed file: GS2W-FtF35B.MP4_full.txt → GS2W-FtF35B_full.txt
Updated title in: GS2W-FtF35B_full.txt
Renamed folder: GS2W-FtF26A.MP4 → GS2W-FtF26A
Renamed file: GS2W-FtF26A.MP4_full.txt → GS2W-FtF26A_full.txt
Updated title in: GS2W-FtF26A_full.txt
Renamed folder: GS2W-FtF33B.MP4 → GS2W-FtF33B
Renamed file: GS2W-FtF33B.MP4_full.txt → GS2W-FtF33B_full.txt
Updated title in: GS2W-FtF33B_full.txt
Renamed folder: GS2W-FtF20B.MP4 → GS2W-FtF20B
Renamed file: GS2W-FtF20B.MP4_full.txt → GS2W-FtF20B_full.txt
Updated ti

------------------------------------------------------------------

In [7]:
import os
import pandas as pd
from io import StringIO

def split_transcript_by_speaker(file_path, output_dir):
    """
    Reads a transcript file with pipe-delimited content and metadata lines (starting with #),
    and writes one output file per speaker role, each with the same metadata header.
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    metadata = [line for line in lines if line.strip().startswith('#')]
    content = [line for line in lines if not line.strip().startswith('#')]

    if not content:
        raise ValueError(f"No transcript content found in: {file_path}")

    df = pd.read_csv(StringIO(''.join(content)), sep='|', header=None, engine='python')
    df.columns = ['Start Time', 'End Time', 'Speaker', 'Speaker Role', 'Transcription']

    # Strip whitespace from Speaker Role for reliable filtering
    df['Speaker Role'] = df['Speaker Role'].str.strip()

    base_name = os.path.splitext(os.path.basename(file_path))[0]
    os.makedirs(output_dir, exist_ok=True)
    base = base_name.split('_')[0]

    for role in df['Speaker Role'].unique():
        role_df = df[df['Speaker Role'] == role]
        role_clean = role.replace(' ', '_')
        out_path = os.path.join(output_dir, f"{base}_{role_clean}.txt")

        with open(out_path, 'w', encoding='utf-8') as out_f:
            # Write shared metadata
            out_f.writelines(metadata)
            # Write the filtered content rows (no header, pipe-separated)
            out_f.write(role_df.to_csv(sep='|', index=False, header=False))

        print(f"  Saved: {out_path} ({len(role_df)} rows)")


def process_directory(input_dir, output_dir):
    """
    Processes all .txt files in input_dir, splitting each by speaker role.
    """
    txt_files = [f for f in os.listdir(input_dir) if f.endswith('.txt')]

    if not txt_files:
        print("No .txt files found in the input directory.")
        return

    for fname in txt_files:
        file_path = os.path.join(input_dir, fname)
        print(f"Processing: {fname}")
        try:
            split_transcript_by_speaker(file_path, output_dir)
        except Exception as e:
            print(f"  ERROR in {fname}: {e}")

    print("\nDone.")


# --- Run it ---
if __name__ == "__main__":
    input_dir = "Test/GSGS#1/dyad"       # folder containing your original files
    output_dir = "Test/GSGS#1/single" # where split files will be saved
    process_directory(input_dir, output_dir)

Processing: GS1-VtV14_dyad.txt
  Saved: Test/GSGS#1/single/GS1-VtV14_Ali.txt (42 rows)
  Saved: Test/GSGS#1/single/GS1-VtV14_Bailey.txt (42 rows)
Processing: GS1-VtV5B_full.txt
  Saved: Test/GSGS#1/single/GS1-VtV5B_Main.txt (8 rows)
  Saved: Test/GSGS#1/single/GS1-VtV5B_RA.txt (7 rows)
  Saved: Test/GSGS#1/single/GS1-VtV5B_Main-2.txt (5 rows)
  Saved: Test/GSGS#1/single/GS1-VtV5B_Ali.txt (16 rows)
  Saved: Test/GSGS#1/single/GS1-VtV5B_Bailey.txt (15 rows)
Processing: GS1-VtV6_dyad.txt
  Saved: Test/GSGS#1/single/GS1-VtV6_Ali.txt (59 rows)
  Saved: Test/GSGS#1/single/GS1-VtV6_Bailey.txt (55 rows)
Processing: GS1-VtV3_dyad.txt
  Saved: Test/GSGS#1/single/GS1-VtV3_Ali.txt (36 rows)
  Saved: Test/GSGS#1/single/GS1-VtV3_Bailey.txt (26 rows)
Processing: GS1-VtV11_dyad.txt
  Saved: Test/GSGS#1/single/GS1-VtV11_Bailey.txt (26 rows)
  Saved: Test/GSGS#1/single/GS1-VtV11_Ali.txt (21 rows)
Processing: GS1-VtV1_dyad.txt
  Saved: Test/GSGS#1/single/GS1-VtV1_Bailey.txt (56 rows)
  Saved: Test/GSGS#1

In [ ]:
import os
import shutil

def collect_txt_files(root_dir, output_dir="full"):
    """
    Recursively finds all .txt files under root_dir and copies them into output_dir.
    Handles filename conflicts by prefixing the subfolder name.
    """
    os.makedirs(output_dir, exist_ok=True)
    copied = 0
    skipped = 0

    for dirpath, dirnames, filenames in os.walk(root_dir):
        for fname in filenames:
            if fname.endswith('.txt'):
                src = os.path.join(dirpath, fname)
                dst = os.path.join(output_dir, fname)

                # If a file with the same name already exists, prefix with subfolder name
                if os.path.exists(dst):
                    subfolder = os.path.basename(dirpath)
                    fname = f"{subfolder}_{fname}"
                    dst = os.path.join(output_dir, fname)

                shutil.copy2(src, dst)
                print(f"Copied: {src} → {dst}")
                copied += 1

    print(f"\nDone. {copied} file(s) copied, {skipped} skipped.")

# --- Run it ---
if __name__ == "__main__":
    root_dir = "Test/GSGS#1/dyad"    # folder containing all the subfolders
    output_dir = "full"             # destination folder
    collect_txt_files(root_dir, output_dir)